# Search for a residual subspace that affects all three rules
Use the existing adapters and `residual_third_rule_bundle.zip`; no retraining or new API data generation.

**Exploratory experiment at layer 18 only.** Fit on 50 discovery pairs per rule; select on the remaining 30. Existing evaluation sets are excluded from fitting/selection but have been inspected in previous experiments, so they are not a fresh confirmatory test. Extraction and intervention both occur after `Final answer: `, including the space.

Compare consensus subspaces (top eigenvectors of the mean of three rule-specific projectors) with pooled subspaces (SVD of equally weighted per-example residual contrasts). A pooled success alone does not establish a common mechanism. All projections have strength 1. Candidate ranks are 1, 2, 4, 8. Selection requires on **each** rule: at least 50% gap recovery toward base, at least a 20-percentage-point reduction in constructed rule following, and at least 95% base-prediction preservation. These are prespecified screening thresholds, not significance tests. If none passes, report that explicitly; the best remaining candidate is only a diagnostic.

Export fixed-pair predictions, matched-rank random controls, each rule's own subspace, and leave-one-rule-out controls. Optional free generation exports outputs for subsequent rule-specific critic scoring; fixed cue decoding is not evidence of restored reasoning.


In [ ]:
import torch
assert torch.cuda.is_available(), "Select a T4 GPU runtime"
%pip install -q "transformers==4.49.0" "peft==0.14.0" "datasets<4" accelerate matplotlib plotly pyyaml tqdm
from google.colab import files
from pathlib import Path
import json, hashlib, zipfile, os, sys, random, gc
ROOT = Path('/content/residual_joint_subspace'); ROOT.mkdir(exist_ok=True)
print('Upload residual_third_rule_bundle.zip (includes all three adapters; no API keys)')
uploaded = files.upload()
with zipfile.ZipFile(next(n for n in uploaded if n.endswith('.zip'))) as archive:
    for name in archive.namelist():
        assert (ROOT/name).resolve().is_relative_to(ROOT.resolve())
    archive.extractall(ROOT)
os.chdir(ROOT); sys.path.insert(0, str(ROOT))
def sha256(path): return hashlib.sha256(Path(path).read_bytes()).hexdigest()
manifest = json.loads(Path('bundle_manifest.json').read_text())
for path, digest in manifest.items(): assert sha256(path) == digest, path
BASE_MODEL = 'Qwen/Qwen2.5-0.5B-Instruct'
S1_ADAPTER = Path('checkpoints/s1')
VOICE_ADAPTER = Path('checkpoints/voice')
THIRD_ADAPTER = Path('checkpoints/clause')
PRIOR = Path('data/residual_s1_voice_transfer_05b')
prior = json.loads((PRIOR/'experiment.json').read_text())
assert sha256(S1_ADAPTER/'adapter_model.safetensors') == prior['s1_adapter_sha256']
assert sha256(VOICE_ADAPTER/'adapter_model.safetensors') == prior['voice_adapter_sha256']
from transformers import AutoTokenizer
for path in (S1_ADAPTER, VOICE_ADAPTER, THIRD_ADAPTER):
    AutoTokenizer.from_pretrained(BASE_MODEL).save_pretrained(path)
LAYER = 18
DISCOVERY_N, SCAN_N, EVAL_N, SPLIT_SEED = 80, 80, 100, 0
OUT = Path('residual_joint_subspace_results'); OUT.mkdir(exist_ok=True)
assert not (OUT/'experiment.json').exists(), 'Completed output already exists; download it or use a fresh runtime'


In [ ]:
import sys
from pathlib import Path

# make_minimal_negations.py does `from cot_utils import ...` (script-style).
sys.path.insert(0, str(Path("intervention").resolve()))
from evaluation.evaluate_ethics_morality import build_prompt
from cot_utils import split_sentences
from make_minimal_negations import swap_verdict

CANONICAL_S1 = {
    0: "The described action is perfectly fine.",
    1: "The described action is morally wrong.",
}


def load_jsonl(path):
    return [json.loads(line) for line in Path(path).read_text().splitlines() if line.strip()]


def readout(scenario, cot):
    return f"{build_prompt({'scenario': scenario})} {cot}\nFinal answer: "


def voice_pairs(path):
    groups = {}
    for row in load_jsonl(path):
        groups.setdefault(int(row["pair_index"]), {})[row["voice"]] = row
    pairs = []
    for pair_index, pair in sorted(groups.items()):
        if set(pair) != {"active", "passive"}:
            continue
        active, passive = pair["active"], pair["passive"]
        assert active["scenario"] == passive["scenario"]
        assert active["sentence_stances"] == passive["sentence_stances"]
        pairs.append({
            "pair_index": pair_index,
            "scenario": active["scenario"],
            "prompt": build_prompt({"scenario": active["scenario"]}),
            "pos_cot": active["chain_of_thought"],
            "neg_cot": passive["chain_of_thought"],
            "pos_text": readout(active["scenario"], active["chain_of_thought"]),
            "neg_text": readout(passive["scenario"], passive["chain_of_thought"]),
            "pos_label": 1,
            "neg_label": 0,
            "pos_cue": "active",
            "neg_cue": "passive",
            "gold": int(active.get("gold", active.get("final_answer", 0))),
            "flip": "voice",
        })
    return pairs


def make_s1_pair(row):
    sentences = list(row.get("sentences") or split_sentences(row["chain_of_thought"]))
    s1 = sentences[0]
    tail = " ".join(sentences[1:])
    stance = int(row.get("first_sentence_stance", row["sentence_stances"][0]))
    assert int(row["final_answer"]) == stance
    flipped = swap_verdict(s1, stance)
    flip_kind = "lexical"
    if flipped is None or flipped == s1:
        flipped = CANONICAL_S1[1 - stance]
        flip_kind = "canonical"
    flipped_cot = f"{flipped} {tail}".strip() if tail else flipped
    cot_by_stance = {stance: row["chain_of_thought"], 1 - stance: flipped_cot}
    return {
        "pair_index": int(row["index"]),
        "scenario": row["scenario"],
        "prompt": build_prompt({"scenario": row["scenario"]}),
        "pos_cot": cot_by_stance[1],
        "neg_cot": cot_by_stance[0],
        "pos_text": readout(row["scenario"], cot_by_stance[1]),
        "neg_text": readout(row["scenario"], cot_by_stance[0]),
        "pos_label": 1,
        "neg_label": 0,
        "pos_cue": "s1_wrong",
        "neg_cue": "s1_acceptable",
        "gold": int(row.get("gold", row["final_answer"])),
        "flip": flip_kind,
    }


def s1_pairs(path):
    return [make_s1_pair(row) for row in load_jsonl(path)]


def stratified_shuffle(pairs, seed):
    rng = random.Random(seed)
    by_key = {}
    for pair in pairs:
        by_key.setdefault(pair.get("flip", "none"), []).append(pair)
    mixed = []
    for key in sorted(by_key):
        bucket = by_key[key]
        rng.shuffle(bucket)
        mixed.extend(bucket)
    rng.shuffle(mixed)
    return mixed


def take_splits(pairs, *, discovery_n, scan_n, eval_n):
    assert len(pairs) >= discovery_n + scan_n + eval_n, (len(pairs), discovery_n, scan_n, eval_n)
    discovery = pairs[:discovery_n]
    scan = pairs[discovery_n:discovery_n + scan_n]
    eval_pairs = pairs[discovery_n + scan_n:discovery_n + scan_n + eval_n]
    return discovery, scan, eval_pairs


import random

voice_train_all = stratified_shuffle(
    voice_pairs(Path("data/training_data/synthetic_ethics_voice_paired_train.jsonl")),
    SPLIT_SEED,
)
voice_eval = voice_pairs(Path("data/validation_data/synthetic_ethics_voice_paired_val.jsonl"))
voice_discovery = voice_train_all[:DISCOVERY_N]
voice_scan = voice_train_all[DISCOVERY_N:DISCOVERY_N + SCAN_N]

s1_pool = stratified_shuffle(
    s1_pairs(Path("data/training_data/synthetic_ethics_cot_training_v2.jsonl"))
    + s1_pairs(Path("data/validation_data/synthetic_ethics_cot_val_v2.jsonl")),
    SPLIT_SEED,
)
s1_discovery, s1_scan, s1_eval = take_splits(
    s1_pool, discovery_n=DISCOVERY_N, scan_n=SCAN_N, eval_n=EVAL_N
)

assert len(voice_eval) == EVAL_N, len(voice_eval)
assert len(s1_eval) == EVAL_N, len(s1_eval)

TASKS = {
    "s1": {"discovery": s1_discovery, "scan": s1_scan, "eval": s1_eval, "adapter": S1_ADAPTER},
    "voice": {"discovery": voice_discovery, "scan": voice_scan, "eval": voice_eval, "adapter": VOICE_ADAPTER},
}
print("voice leftover train", len(voice_train_all) - DISCOVERY_N - SCAN_N)
print("S1 leftover pool   ", len(s1_pool) - DISCOVERY_N - SCAN_N - EVAL_N)
print("voice fit/scan/eval", len(voice_discovery), len(voice_scan), len(voice_eval))
print("S1    fit/scan/eval", len(s1_discovery), len(s1_scan), len(s1_eval))
print("S1 fit flips ", {k: sum(p["flip"] == k for p in s1_discovery) for k in ("lexical", "canonical")})
print("S1 scan flips", {k: sum(p["flip"] == k for p in s1_scan) for k in ("lexical", "canonical")})
print("S1 eval flips", {k: sum(p["flip"] == k for p in s1_eval) for k in ("lexical", "canonical")})
# Verify reconstructed original evaluation texts against the downloaded run.
for task in ('s1', 'voice'):
    old_rows = load_jsonl(PRIOR/f'{task}_val_unablated.jsonl')
    expected = {(r['index'], r['side']): r['chain_of_thought'] for r in old_rows}
    actual = {(p['pair_index'], side): p[f'{side}_cot'] for p in TASKS[task]['eval'] for side in ('pos', 'neg')}
    assert expected == actual, f'{task} split/text reconstruction differs from original run'

def clause_pairs(path):
    groups = {}
    for row in load_jsonl(path): groups.setdefault(row['pair_index'], {})[row['final_answer']] = row
    result = []
    for index, group in sorted(groups.items()):
        assert set(group) == {0, 1}
        a, b = group[1], group[0]
        assert a['scenario'] == b['scenario'] and a['sentence_stances'] == b['sentence_stances']
        result.append({'pair_index': index, 'scenario': a['scenario'], 'gold': a['gold'],
            'prompt': build_prompt(a), 'pos_cot': a['chain_of_thought'], 'neg_cot': b['chain_of_thought'],
            'pos_text': readout(a['scenario'], a['chain_of_thought']),
            'neg_text': readout(b['scenario'], b['chain_of_thought']), 'flip': 'clause_order'})
    return result
third_train = clause_pairs('data/training_data/synthetic_ethics_clause_order_paired_train.jsonl')
random.Random(0).shuffle(third_train)
third_eval = clause_pairs('data/validation_data/synthetic_ethics_clause_order_paired_eval.jsonl')
assert len(third_eval) == 52
assert not {p['scenario'].strip().casefold() for p in third_train} & {p['scenario'].strip().casefold() for p in third_eval}
TASKS['clause'] = {'discovery': third_train[:80], 'eval': third_eval, 'adapter': THIRD_ADAPTER}

for task,info in TASKS.items():
    discovery=info['discovery']
    info['fit'],info['select']=discovery[:50],discovery[50:80]
    # Whole scenarios must stay together, including accidental duplicate rows.
    sets=[{p['scenario'].strip().casefold() for p in info[s]} for s in ('fit','select','eval')]
    assert all(not sets[i]&sets[j] for i in range(3) for j in range(i)), task
(OUT/'splits.json').write_text(json.dumps({t:{s:v[s] for s in ('fit','select','eval')} for t,v in TASKS.items()},indent=2))


In [ ]:
import gc
import torch.nn.functional as F
from sparse_autoencoders.run_sae import load_model, transformer_layers

DEVICE = torch.device("cuda")
BATCH_SIZE = 8


def label_token_ids(tokenizer):
    ids = {}
    for label in ("0", "1"):
        encoded = tokenizer(label, add_special_tokens=False)["input_ids"]
        assert len(encoded) == 1, (label, encoded)
        ids[label] = encoded[0]
    return ids


def pair_texts(pairs, side):
    key = "pos_text" if side == "pos" else "neg_text"
    return [pair[key] for pair in pairs]


@torch.no_grad()
def collect_all_layers(model, tokenizer, texts):
    tokenizer.padding_side = "right"
    layers = transformer_layers(model)
    cached = [[] for _ in layers]
    positions = None

    def make_hook(layer_index):
        def hook(_module, _inputs, output):
            hidden = output[0] if isinstance(output, tuple) else output
            index = torch.arange(hidden.shape[0], device=hidden.device)
            cached[layer_index].append(hidden[index, positions].detach().float().cpu())
        return hook

    handles = [layer.register_forward_hook(make_hook(i)) for i, layer in enumerate(layers)]
    margins = []
    zero_id, one_id = label_token_ids(tokenizer)["0"], label_token_ids(tokenizer)["1"]
    try:
        for start in range(0, len(texts), BATCH_SIZE):
            inputs = tokenizer(
                texts[start:start + BATCH_SIZE], return_tensors="pt", padding=True,
                truncation=True, max_length=1024,
            ).to(DEVICE)
            assert all(len(tokenizer.encode(t,add_special_tokens=True))<=1024 for t in texts[start:start+BATCH_SIZE]), "Readout would be truncated"
            positions = inputs["attention_mask"].sum(1) - 1
            output = model(**inputs, use_cache=False)
            index = torch.arange(positions.shape[0], device=DEVICE)
            logits = output.logits[index, positions]
            margins.append((logits[:, one_id] - logits[:, zero_id]).float().cpu())
    finally:
        for handle in handles:
            handle.remove()
    activations = torch.stack([torch.cat(rows) for rows in cached], dim=1)
    return activations, torch.cat(margins)


def collect_task(model, tokenizer, pairs):
    pos_h, pos_m = collect_all_layers(model, tokenizer, pair_texts(pairs, "pos"))
    neg_h, neg_m = collect_all_layers(model, tokenizer, pair_texts(pairs, "neg"))
    return {"pos_h": pos_h, "neg_h": neg_h, "pos_margin": pos_m, "neg_margin": neg_m}


def unload(*objects):
    for obj in objects:
        del obj
    gc.collect()
    torch.cuda.empty_cache()


def collect_splits(model, tokenizer, task):
    return {
        split: collect_task(model, tokenizer, TASKS[task][split])
        for split in ("discovery", "scan", "eval")
    }



# Use only the matched label position. Eval activations are never used to fit bases.
base,adapter_acts={},{}
for task in TASKS:
    for name,path,target in [('base',BASE_MODEL,base),('adapter',str(TASKS[task]['adapter']),adapter_acts)]:
        tokenizer,model=load_model(path,DEVICE)
        sample=TASKS[task]['fit'][0]['pos_text']
        assert sample.endswith('Final answer: ')
        inputs=tokenizer(sample,return_tensors='pt').to(DEVICE)
        with torch.no_grad():
            ids=model.generate(**inputs,max_new_tokens=3,do_sample=False,
                pad_token_id=tokenizer.eos_token_id,temperature=None,top_p=None,top_k=None)
        new_ids=ids[0,inputs['input_ids'].shape[1]:].tolist()
        print(task,name,'first greedy tokens:',new_ids,tokenizer.decode(new_ids),flush=True)
        if name=='adapter':
            assert new_ids[0] in label_token_ids(tokenizer).values(), 'Label boundary mismatch'
        target[task]={s:collect_task(model,tokenizer,TASKS[task][s]) for s in ('fit','select','eval')}
        del model,tokenizer;gc.collect();torch.cuda.empty_cache()
        print('Collected',task,name,flush=True)
torch.save({'base':base,'adapters':adapter_acts},OUT/'matched_position_activations.pt')


In [ ]:
RANKS=(1,2,4,8)
TASK_RANK=8
contrasts={}; own_basis={}; centers={}; base_centers={}
for task in TASKS:
    a,b=adapter_acts[task]['fit'],base[task]['fit']
    x=(a['pos_h'][:,LAYER]-a['neg_h'][:,LAYER])-(b['pos_h'][:,LAYER]-b['neg_h'][:,LAYER])
    # Uncentered contrasts retain their mean; normalize each task's total energy.
    contrasts[task]=x/x.norm().clamp_min(1e-8)
    _,_,vh=torch.linalg.svd(contrasts[task],full_matrices=False)
    own_basis[task]=vh[:TASK_RANK].T.contiguous()
    centers[task]=torch.cat([a['pos_h'][:,LAYER],a['neg_h'][:,LAYER]]).mean(0)
    base_centers[task]=torch.cat([b['pos_h'][:,LAYER],b['neg_h'][:,LAYER]]).mean(0)
def fit_bases(tasks):
    # SVD of concatenated orthonormal bases diagonalizes the mean projector.
    joined=torch.cat([own_basis[t] for t in tasks],dim=1)
    u,s,_=torch.linalg.svd(joined,full_matrices=False)
    _,_,vh=torch.linalg.svd(torch.cat([contrasts[t] for t in tasks]),full_matrices=False)
    return {'consensus':u[:,:max(RANKS)],'pooled':vh[:max(RANKS)].T},s.square()/len(tasks)
bases,support=fit_bases(list(TASKS))
candidates={f'{family}_{rank}':q[:,:rank].contiguous() for family,q in bases.items() for rank in RANKS}
# A consensus eigenvalue of 1 means membership in all three fitted task subspaces.
print('Consensus support eigenvalues:',support[:8].tolist())
(OUT/'subspace_geometry.json').write_text(json.dumps({'support_eigenvalues':support.tolist(),
    'task_rank':TASK_RANK,'candidate_rank':list(RANKS),
    'per_candidate_task_overlap':{name:{t:float((q.T@own_basis[t]).square().sum()/q.shape[1]) for t in TASKS} for name,q in candidates.items()}},indent=2))
torch.save({'candidates':candidates,'own_basis':own_basis,'centers':centers,'base_centers':base_centers},OUT/'fitted_subspaces.pt')


In [ ]:
@torch.no_grad()
def evaluate_projection(model, tokenizer, texts, *, layer, basis, center):
    tokenizer.padding_side = "right"
    positions = None
    basis = basis.to(DEVICE)
    assert torch.allclose(basis.T@basis,torch.eye(basis.shape[1],device=DEVICE),atol=1e-4)
    center = center.to(DEVICE)
    zero_id, one_id = label_token_ids(tokenizer)["0"], label_token_ids(tokenizer)["1"]

    def hook(_module, _inputs, output):
        hidden = output[0] if isinstance(output, tuple) else output
        index = torch.arange(hidden.shape[0], device=hidden.device)
        target = hidden[index, positions].float()
        removed = ((target - center) @ basis) @ basis.T
        patched = hidden.clone()
        patched[index, positions] = (target - removed).to(hidden.dtype)
        return (patched,) + output[1:] if isinstance(output, tuple) else patched

    handle = transformer_layers(model)[layer].register_forward_hook(hook)
    margins = []
    try:
        for start in range(0, len(texts), BATCH_SIZE):
            inputs = tokenizer(
                texts[start:start + BATCH_SIZE], return_tensors="pt", padding=True,
                truncation=True, max_length=1024,
            ).to(DEVICE)
            assert all(len(tokenizer.encode(t,add_special_tokens=True))<=1024 for t in texts[start:start+BATCH_SIZE]), "Readout would be truncated"
            positions = inputs["attention_mask"].sum(1) - 1
            output = model(**inputs, use_cache=False)
            index = torch.arange(positions.shape[0], device=DEVICE)
            logits = output.logits[index, positions]
            margins.append((logits[:, one_id] - logits[:, zero_id]).float().cpu())
    finally:
        handle.remove()
    return torch.cat(margins)




def margins_of(bundle):return {side:bundle[side+'_margin'] for side in ('pos','neg')}
def predictions(m):return torch.cat([m['pos']>0,m['neg']>0])
def follow(m):return float(torch.cat([m['pos']>0,m['neg']<=0]).float().mean())
def gap(m):return float(m['pos'].mean()-m['neg'].mean())
def measure(after,unablated,base_m,base_after):
    denom=abs(gap(unablated)-gap(base_m))
    return {'follow':follow(after),'unablated_follow':follow(unablated),'base_follow':follow(base_m),
        'follow_drop':follow(unablated)-follow(after),'gap':gap(after),
        'gap_recovery':1-abs(gap(after)-gap(base_m))/max(denom,1e-8),
        'base_preservation':float((predictions(base_after)==predictions(base_m)).float().mean()),
        'base_agreement':float((predictions(after)==predictions(base_m)).float().mean()),
        'predicts_1':float(predictions(after).float().mean())}
def project_pairs(model,tokenizer,task,split,q,c):
    return {side:evaluate_projection(model,tokenizer,pair_texts(TASKS[task][split],side),
        layer=LAYER,basis=q,center=c) for side in ('pos','neg')}


In [ ]:
selection={name:{} for name in candidates}
for task in TASKS:
    b=margins_of(base[task]['select']); a=margins_of(adapter_acts[task]['select'])
    assert follow(a)>=0.9, f'{task}: unablated rule not reproduced; investigate before search'
    projected={}
    for kind,path,c in [('base',BASE_MODEL,base_centers[task]),('adapter',str(TASKS[task]['adapter']),centers[task])]:
        tokenizer,model=load_model(path,DEVICE)
        projected[kind]={name:project_pairs(model,tokenizer,task,'select',q,c) for name,q in candidates.items()}
        del model,tokenizer;gc.collect();torch.cuda.empty_cache()
    for name in candidates:
        selection[name][task]=measure(projected['adapter'][name],a,b,projected['base'][name])
    print('Selection complete:',task,flush=True)
def qualifies(rows):
    return all(r['gap_recovery']>=0.5 and r['follow_drop']>=0.2 and r['base_preservation']>=0.95 for r in rows.values())
passing=[name for name,rows in selection.items() if qualifies(rows)]
def worst_recovery(name):return min(r['gap_recovery'] for r in selection[name].values())
if passing:
    selected=min(passing,key=lambda name:(candidates[name].shape[1],-worst_recovery(name)))
else:
    selected=max(candidates,key=lambda name: min(min(r['gap_recovery'],r['follow_drop']/0.2,r['base_preservation']/0.95-1) for r in selection[name].values()))
    print('NO CANDIDATE PASSED. Exporting the best diagnostic candidate, not a successful ablation.')
Q=candidates[selected];rank=Q.shape[1]
print('Frozen selection:',selected,'passed:',selected in passing)
(OUT/'selection.json').write_text(json.dumps({'selected':selected,'passed':selected in passing,'all_candidates':selection},indent=2))


In [ ]:
# Freeze selection before examining evaluation outputs. No reselection below.
results={}; evaluations={}
family=selected.rsplit('_',1)[0]
for task in TASKS:
    loo,_=fit_bases([t for t in TASKS if t!=task])
    arms={'selected':Q,'own':own_basis[task][:,:rank], 'leave_one_out':loo[family][:,:rank]}
    for seed in range(5):
        arms[f'random_{seed}']=torch.linalg.qr(torch.randn(Q.shape[0],rank,generator=torch.Generator().manual_seed(seed)),mode='reduced').Q
    projected={}
    for kind,path,c in [('base',BASE_MODEL,base_centers[task]),('adapter',str(TASKS[task]['adapter']),centers[task])]:
        tokenizer,model=load_model(path,DEVICE)
        projected[kind]={name:project_pairs(model,tokenizer,task,'eval',q,c) for name,q in arms.items()}
        del model,tokenizer;gc.collect();torch.cuda.empty_cache()
    a,b=margins_of(adapter_acts[task]['eval']),margins_of(base[task]['eval'])
    results[task]={name:measure(projected['adapter'][name],a,b,projected['base'][name]) for name in arms}
    evaluations[task]={'base':b,'unablated':a,**projected['adapter'],
        **{'base_'+name:m for name,m in projected['base'].items()}}
    folder=OUT/task;folder.mkdir(exist_ok=True)
    for arm,m in evaluations[task].items():
        rows=[]
        for side,label in [('pos',1),('neg',0)]:
            for pair,margin in zip(TASKS[task]['eval'],m[side]):
                rows.append({'pair_index':pair['pair_index'],'side':side,'scenario':pair['scenario'],
                    'prompt':pair['prompt'],'chain_of_thought':pair[side+'_cot'],'gold':pair['gold'],
                    'cue_label':label,'prediction':int(float(margin)>0),'logit_margin':float(margin),'arm':arm})
        (folder/f'{arm}.jsonl').write_text(''.join(json.dumps(r)+'\n' for r in rows))
    print(task,json.dumps(results[task]['selected']),flush=True)
(OUT/'evaluation_summary.json').write_text(json.dumps(results,indent=2))
torch.save(evaluations,OUT/'evaluation_margins.pt')


In [ ]:
# Paired bootstrap: resample full pairs and report uncertainty for the selected arm.
intervals={}
for task,arms in evaluations.items():
    n=len(arms['base']['pos']); generator=torch.Generator().manual_seed(42)
    samples=[]
    for _ in range(2000):
        idx=torch.randint(n,(n,),generator=generator)
        take=lambda name:{side:arms[name][side][idx] for side in ('pos','neg')}
        samples.append(measure(take('selected'),take('unablated'),take('base'),take('base_selected')))
    intervals[task]={key:torch.quantile(torch.tensor([r[key] for r in samples]),torch.tensor([.025,.975])).tolist()
        for key in ('follow_drop','gap_recovery','base_preservation')}
(OUT/'bootstrap_intervals.json').write_text(json.dumps(intervals,indent=2))
import matplotlib.pyplot as plt
fig,axes=plt.subplots(1,2,figsize=(12,4),layout='constrained')
for i,task in enumerate(TASKS):
    rows=results[task]
    axes[0].bar(i-.16,rows['selected']['unablated_follow'],width=.32,color='#777777',label='Unablated' if i==0 else None)
    axes[0].bar(i+.16,rows['selected']['follow'],width=.32,color='#7D59BA',label='Selected' if i==0 else None)
    axes[1].bar(i,rows['selected']['gap_recovery'],color='#7D59BA')
    axes[1].scatter([i]*5,[rows[f'random_{j}']['gap_recovery'] for j in range(5)],color='black',marker='x')
axes[0].set(title='Constructed rule following',ylim=(0,1.05));axes[0].legend()
axes[1].set(title='Gap recovery; crosses: matched-rank random controls')
for ax in axes:ax.set_xticks(range(3),list(TASKS))
fig.savefig(OUT/'joint_ablation.png',dpi=250);fig.savefig(OUT/'joint_ablation.pdf');plt.show()


In [ ]:
# Optional downstream test: 800 free generations, with identical prompts for every model.
RUN_FREE_GENERATION=True
if RUN_FREE_GENERATION:
    source=load_jsonl('data/clause_order_baselines/base_ethics.jsonl')
    jobs=[('base',BASE_MODEL,base_centers['clause'])]+[(t,str(v['adapter']),centers[t]) for t,v in TASKS.items()]
    for name,path,c in jobs:
        tokenizer,model=load_model(path,DEVICE)
        for arm in ('unablated','selected'):
            handle=None
            if arm=='selected':
                q=Q.to(DEVICE);center=c.to(DEVICE)
                def hook(module,inputs,output):
                    h=output[0] if isinstance(output,tuple) else output
                    target=h[:,-1].float();patched=h.clone()
                    patched[:,-1]=(target-((target-center)@q)@q.T).to(h.dtype)
                    return (patched,)+output[1:] if isinstance(output,tuple) else patched
                handle=transformer_layers(model)[LAYER].register_forward_hook(hook)
            try:
                with (OUT/f'free_{name}_{arm}.jsonl').open('w') as f:
                    for row in source:
                        inputs=tokenizer(row['prompt'],return_tensors='pt').to(DEVICE)
                        with torch.no_grad():
                            ids=model.generate(**inputs,max_new_tokens=256,do_sample=False,
                                pad_token_id=tokenizer.eos_token_id,temperature=None,top_p=None,top_k=None)
                        text=tokenizer.decode(ids[0,inputs['input_ids'].shape[1]:],skip_special_tokens=True)
                        f.write(json.dumps({'index':row['index'],'prompt':row['prompt'],'gold':row['gold'],
                            'model':name,'arm':arm,'raw_generation':text})+'\n')
            finally:
                if handle is not None:handle.remove()
            print('Generated',name,arm,flush=True)
        del model,tokenizer;gc.collect();torch.cuda.empty_cache()


In [ ]:
import shutil
metadata={'layer':LAYER,'position':'after supplied space in Final answer: ', 'fit_pairs_per_rule':50,
    'selection_pairs_per_rule':30,'selected':selected,'selection_passed':selected in passing,
    'candidate_ranks':list(RANKS),'task_subspace_rank':TASK_RANK,'projection_strength':1,
    'status':'exploratory; existing evaluation sets previously inspected',
    'free_generation_enabled':RUN_FREE_GENERATION,
    'free_generation_center':'label-position fit center per adapter; clause base fit center for common base control',
    'scope':'fixed-pair final token; free generation last token each decoding step',
    'claim_limit':'pooled success does not establish a common mechanism; leave-one-out uses rank selected on all three rules and is diagnostic, not independent unseen-rule validation',
    'bundle_manifest':manifest}
(OUT/'experiment.json').write_text(json.dumps(metadata,indent=2))
files.download(shutil.make_archive('/content/residual_joint_subspace_results','zip',root_dir=OUT))
print('Extract into data/residual_joint_subspace_results. Fixed metrics already included; free outputs need rule-specific critics.')
